In [ ]:
# --- Headless Chrome setup that works on GitHub Actions and locally ---
import os, shutil
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import logging

OUTDIR = "outputs"
os.makedirs(OUTDIR, exist_ok=True)


# Configure logging once
logging.basicConfig(
    level=logging.INFO,  # change to DEBUG if you want more detail
    format="%(asctime)s [%(levelname)s] %(message)s"
)

logger = logging.getLogger(__name__)

opts = Options()
opts.add_argument("--headless=new")          # required on CI
opts.add_argument("--no-sandbox")            # required on CI
opts.add_argument("--disable-dev-shm-usage") # required on CI
opts.add_argument("--disable-gpu")
# unique profile per run to avoid "user data directory is already in use"
opts.add_argument(f"--user-data-dir=/tmp/chrome-profile-{os.getenv('RUN_ID','local')}")

# Use chromedriver installed on the runner (if present), else let Selenium manage locally
drv_path = shutil.which("chromedriver")
service = Service(drv_path) if drv_path else None
browser = webdriver.Chrome(service=service, options=opts)




In [ ]:
## Packages

from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import glob
import os
import time
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta

In [ ]:
## Functions
summary_ = None
#Does the variable exist (by xpath)
def check_exists_by_xpath(xpath):
    time.sleep(1)
    try:
        browser.find_element("xpath",xpath)
    except NoSuchElementException:
        return False
    return True

#Does the variable exist (by name)
def CheckExistsByName(find_name):
    time.sleep(1)
    try:
        browser.find_element("name",find_name)
    except NoSuchElementException:
        return False
    return True

#Scrape data from day summary page
def summary_page_scrape(input_date):
    row_list = [2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21]
    case_type_xpath = []
    case_defendant_xpath = []
    case_plaintiff_xpath = []
    case_number_xpath = []
    case_hearing_time_xpath = []
    for i in row_list:
        if check_exists_by_xpath("/html/body/table[1]/tbody/tr[1]/td/table/tbody/tr[2]/td[2]/form/table/tbody/tr[3]/td/table/tbody/tr["+ str(i) +"]/td[5]") == True:
            case_type_xpath.append("/html/body/table[1]/tbody/tr[1]/td/table/tbody/tr[2]/td[2]/form/table/tbody/tr[3]/td/table/tbody/tr["+ str(i) +"]/td[5]")
            case_defendant_xpath.append("/html/body/table[1]/tbody/tr[1]/td/table/tbody/tr[2]/td[2]/form/table/tbody/tr[3]/td/table/tbody/tr["+ str(i) +"]/td[3]")
            case_plaintiff_xpath.append("/html/body/table[1]/tbody/tr[1]/td/table/tbody/tr[2]/td[2]/form/table/tbody/tr[3]/td/table/tbody/tr["+ str(i) +"]/td[4]")
            case_hearing_time_xpath.append("/html/body/table[1]/tbody/tr[1]/td/table/tbody/tr[2]/td[2]/form/table/tbody/tr[3]/td/table/tbody/tr["+ str(i) +"]/td[6]")
            case_number_xpath.append("/html/body/table[1]/tbody/tr[1]/td/table/tbody/tr[2]/td[2]/form/table/tbody/tr[3]/td/table/tbody/tr["+ str(i) +"]/td[2]/a")
    case_type = []
    for case in case_type_xpath:
        case_type_i = browser.find_element("xpath",case).text
        case_type.append(case_type_i)
    case_number = []
    for case in case_number_xpath:
        case_number_i = browser.find_element("xpath",case).text
        case_number.append(case_number_i)
    case_defendant = []
    for case in case_defendant_xpath:
        case_defendant_i = browser.find_element("xpath",case).text
        case_defendant.append(case_defendant_i)
    case_plaintiff = []
    for case in case_plaintiff_xpath:
        case_plaintiff_i = browser.find_element("xpath",case).text
        case_plaintiff.append(case_plaintiff_i)
    case_hearing_time = []
    for case in case_hearing_time_xpath:
        case_hearing_time_i = browser.find_element("xpath",case).text
        case_hearing_time.append(case_hearing_time_i)
    d = {'CaseNumber':case_number,'Defendant':case_defendant,'Plaintiff':case_plaintiff,'CaseType':case_type, 'HearingTime':case_hearing_time}
    df = pd.DataFrame(d)
    df['HearingDate'] = input_date
    input_date_clean = input_date.replace("/","-")
    # df.to_csv('summary_'+input_date_clean+"_"+ str(k) +".csv", index = False)  
    # df.to_csv('summary_'+input_date_clean+"_"+ str(k) +".csv", index=False)
    df.to_csv(os.path.join(OUTDIR, f"summary_{input_date_clean}_{k}.csv"), index=False)

    summmary_ = df

def deletefiles(path):
    # files_in_directory = os.listdir(path)
    # filtered_files = [file for file in files_in_directory if file.endswith(".csv")]
    # for file in filtered_files:
    #     path_to_file = os.path.join(path, file)
    #     os.remove(path_to_file)
    logger.info("Will not delete, haha!")
    

In [ ]:
#browser = webdriver.Chrome()
logger.info("Started")

browser.get('https://eapps.courts.state.va.us/gdcourts')
time.sleep(3)
#Accept the Terms and Conditions of Use
elem = browser.find_element("xpath",'/html/body/table/tbody/tr[1]/td/table/tbody/tr[3]/td/form/table/tbody/tr[2]/td/input[1]').click()  # Find the search box
time.sleep(3)

#Manual date(s) set if needed - code below automatically searches previous six weeks and six weeks forward 
input_dates = []

curr_date = pd.to_datetime('today').date()

start_date = curr_date - relativedelta(days = 84)
#start = datetime.datetime.strptime(start_date, "%d-%m-%Y")
date_generated = pd.date_range(start_date, periods = 140)
#date_generated = pd.date_range(start='06/29/2022', end='06/30/2022')


input_dates = list(date_generated.strftime("%m/%d/%Y"))

print(input_dates)
logger.info(input_dates)

for input_date in input_dates:
    #Go to General Court of Alexandria
    browser.get('https://eapps.courts.state.va.us/gdcourts/caseSearch.do?fromSidebar=true&searchLanding=searchLanding&searchType=hearingDate&searchDivision=V&searchFipsCode=510&curentFipsCode=510')
    #time.sleep(3)
    browser.find_element("xpath",'/html/body/table[1]/tbody/tr[1]/td/table/tbody/tr[2]/td[2]/form/table/tbody/tr/td/table/tbody/tr[2]/td/table/tbody/tr[1]/td[2]/input').send_keys(input_date)
    search = browser.find_element("xpath",'/html/body/table[1]/tbody/tr[1]/td/table/tbody/tr[2]/td[2]/form/table/tbody/tr/td/table/tbody/tr[3]/td/input[1]').click()

    k = 1
    
    while CheckExistsByName("caseInfoScrollForward") == True:
        summary_page_scrape(input_date)
        k = k+1
        browser.find_element("name","caseInfoScrollForward").click()
        logger.info('still inside the while loop')
    else:
        summary_page_scrape(input_date)
    

['06/21/2025', '06/22/2025', '06/23/2025', '06/24/2025', '06/25/2025', '06/26/2025', '06/27/2025', '06/28/2025', '06/29/2025', '06/30/2025', '07/01/2025', '07/02/2025', '07/03/2025', '07/04/2025', '07/05/2025', '07/06/2025', '07/07/2025', '07/08/2025', '07/09/2025', '07/10/2025', '07/11/2025', '07/12/2025', '07/13/2025', '07/14/2025', '07/15/2025', '07/16/2025', '07/17/2025', '07/18/2025', '07/19/2025', '07/20/2025', '07/21/2025', '07/22/2025', '07/23/2025', '07/24/2025', '07/25/2025', '07/26/2025', '07/27/2025', '07/28/2025', '07/29/2025', '07/30/2025', '07/31/2025', '08/01/2025', '08/02/2025', '08/03/2025', '08/04/2025', '08/05/2025', '08/06/2025', '08/07/2025', '08/08/2025', '08/09/2025', '08/10/2025', '08/11/2025', '08/12/2025', '08/13/2025', '08/14/2025', '08/15/2025', '08/16/2025', '08/17/2025', '08/18/2025', '08/19/2025', '08/20/2025', '08/21/2025', '08/22/2025', '08/23/2025', '08/24/2025', '08/25/2025', '08/26/2025', '08/27/2025', '08/28/2025', '08/29/2025', '08/30/2025', '08/3

In [ ]:
# Read in files to combine into one data frame
# Code below to delete the temporary files in the folder

all_files = glob.glob(os.path.join(OUTDIR, "summary_*.csv"))

li = [pd.read_csv(f) for f in all_files]
frame = pd.concat(li, axis = 0, ignore_index=True)

# Normalize column names to lowercase with underscores
frame.columns = frame.columns.str.lower().str.replace(' ', '_')

# Format hearingdate to YYYY-MM-DD format
frame['hearingdate'] = pd.to_datetime(frame['hearingdate'], errors='coerce')
frame['hearingdate'] = frame['hearingdate'].dt.strftime('%Y-%m-%d')


In [ ]:
# Filter to just Unlawful Detainers
UD_days = frame[frame.casetype.str.contains('Unlawful', na=False)]

# Format hearingdate to YYYY-MM-DD format
UD_days['hearingdate'] = pd.to_datetime(UD_days['hearingdate'], errors='coerce')
UD_days['hearingdate'] = UD_days['hearingdate'].dt.strftime('%Y-%m-%d')

# Number of hearings for the date range
len(UD_days.index)
#print(UD_days)


1677

In [ ]:
# Scrape information from individual case number pages

# Go to Case Number Search (Set to Alexandria FIPS)
browser.get('https://eapps.courts.state.va.us/gdcourts/criminalCivilCaseSearch.do?fromSidebar=true&formAction=searchLanding&searchDivision=V&searchFipsCode=510&curentFipsCode=510')

# Convert Case Numbers from UD_days to List for Case Number iteration search 
case_num_list_repeat = UD_days['casenumber'].tolist()
# Make the list unique in case there are repeat case numbers
case_num_array = np.array(case_num_list_repeat)
case_num_array_dist = np.unique(case_num_array)
case_num_list = case_num_array_dist.tolist()


In [ ]:
# Initialize empty lists for all case detail fields
CaseNumber = []
FiledDate = []
CaseType = []
DebtType = []
Judgment = []
Costs = []
AttorneyFees = []
PrincipalAmount = []
OtherAmount = []
InterestAward = []
Possession = []
WritIssuedDate = []
HomesteadExemptionWaived = []
IsJudgmentSatisfied = []
DateSatisfactionFiled = []
OtherAwarded = []
FurtherCaseInformation = []
Garnishee = []
Address = []
GarnisheeAnswer = []
AnswerDate = []
NumberofChecksReceived = []
AppealDate = []
AppealedBy = []
WritofEvictionIssuedDate = []
WritofFieriFaciasIssuedDate = []
Plaintiff1Name = []
Plaintiff1DBATA = []
Plaintiff1Address = []
Plaintiff1Attorney = []
Plaintiff2Name = []
Plaintiff2DBATA = []
Plaintiff2Address = []
Plaintiff2Attorney = []
Plaintiff3Name = []
Plaintiff3DBATA = []
Plaintiff3Address = []
Plaintiff3Attorney = []
Defendant1Name = []
Defendant1DBATA = []
Defendant1Address = []
Defendant1Attorney = []
Defendant2Name = []
Defendant2DBATA = []
Defendant2Address = []
Defendant2Attorney = []
Defendant3Name = []
Defendant3DBATA = []
Defendant3Address = []
Defendant3Attorney = []
Defendant4Name = []
Defendant4DBATA = []
Defendant4Address = []
Defendant4Attorney = []
Defendant5Name = []
Defendant5DBATA = []
Defendant5Address = []
Defendant5Attorney = []
Defendant6Name = []
Defendant6DBATA = []
Defendant6Address = []
Defendant6Attorney = []
Next_Hearing_Date = []
Next_Hearing_Time = []
Admin_tag = []

# Loop through each unique case number and scrape details
for case_num in case_num_list:
    logger.info(f"Scraping details for case: {case_num}")
    
    # Clear the search field and input the case number
    search_field = browser.find_element("xpath", '/html/body/table[1]/tbody/tr[1]/td/table/tbody/tr[2]/td[2]/form/table/tbody/tr[1]/td[2]/input')  # Adjust XPath for case number input
    search_field.clear()
    search_field.send_keys(case_num)
    
    # Click the search button
    search_button = browser.find_element("xpath", '/html/body/table[1]/tbody/tr[1]/td/table/tbody/tr[2]/td[2]/form/table/tbody/tr[2]/td/input[1]')  # Adjust XPath for search button
    search_button.click()
    
    time.sleep(3)  # Wait for page to load
    
    # Scrape each field (replace XPaths with actual ones from the case detail page)
    try:
        CaseNumber.append(browser.find_element("xpath", "//xpath/to/caseNumber").text if check_exists_by_xpath("//xpath/to/caseNumber") else "")
        FiledDate.append(browser.find_element("xpath", "//xpath/to/filedDate").text if check_exists_by_xpath("//xpath/to/filedDate") else "")
        CaseType.append(browser.find_element("xpath", "//xpath/to/caseType").text if check_exists_by_xpath("//xpath/to/caseType") else "")
        DebtType.append(browser.find_element("xpath", "//xpath/to/debtType").text if check_exists_by_xpath("//xpath/to/debtType") else "")
        Judgment.append(browser.find_element("xpath", "//xpath/to/judgment").text if check_exists_by_xpath("//xpath/to/judgment") else "")
        Costs.append(browser.find_element("xpath", "//xpath/to/costs").text if check_exists_by_xpath("//xpath/to/costs") else "")
        AttorneyFees.append(browser.find_element("xpath", "//xpath/to/attorneyFees").text if check_exists_by_xpath("//xpath/to/attorneyFees") else "")
        PrincipalAmount.append(browser.find_element("xpath", "//xpath/to/principalAmount").text if check_exists_by_xpath("//xpath/to/principalAmount") else "")
        OtherAmount.append(browser.find_element("xpath", "//xpath/to/otherAmount").text if check_exists_by_xpath("//xpath/to/otherAmount") else "")
        InterestAward.append(browser.find_element("xpath", "//xpath/to/interestAward").text if check_exists_by_xpath("//xpath/to/interestAward") else "")
        Possession.append(browser.find_element("xpath", "//xpath/to/possession").text if check_exists_by_xpath("//xpath/to/possession") else "")
        WritIssuedDate.append(browser.find_element("xpath", "//xpath/to/writIssuedDate").text if check_exists_by_xpath("//xpath/to/writIssuedDate") else "")
        HomesteadExemptionWaived.append(browser.find_element("xpath", "//xpath/to/homesteadExemptionWaived").text if check_exists_by_xpath("//xpath/to/homesteadExemptionWaived") else "")
        IsJudgmentSatisfied.append(browser.find_element("xpath", "//xpath/to/isJudgmentSatisfied").text if check_exists_by_xpath("//xpath/to/isJudgmentSatisfied") else "")
        DateSatisfactionFiled.append(browser.find_element("xpath", "//xpath/to/dateSatisfactionFiled").text if check_exists_by_xpath("//xpath/to/dateSatisfactionFiled") else "")
        OtherAwarded.append(browser.find_element("xpath", "//xpath/to/otherAwarded").text if check_exists_by_xpath("//xpath/to/otherAwarded") else "")
        FurtherCaseInformation.append(browser.find_element("xpath", "//xpath/to/furtherCaseInformation").text if check_exists_by_xpath("//xpath/to/furtherCaseInformation") else "")
        Garnishee.append(browser.find_element("xpath", "//xpath/to/garnishee").text if check_exists_by_xpath("//xpath/to/garnishee") else "")
        Address.append(browser.find_element("xpath", "//xpath/to/address").text if check_exists_by_xpath("//xpath/to/address") else "")
        GarnisheeAnswer.append(browser.find_element("xpath", "//xpath/to/garnisheeAnswer").text if check_exists_by_xpath("//xpath/to/garnisheeAnswer") else "")
        AnswerDate.append(browser.find_element("xpath", "//xpath/to/answerDate").text if check_exists_by_xpath("//xpath/to/answerDate") else "")
        NumberofChecksReceived.append(browser.find_element("xpath", "//xpath/to/numberofChecksReceived").text if check_exists_by_xpath("//xpath/to/numberofChecksReceived") else "")
        AppealDate.append(browser.find_element("xpath", "//xpath/to/appealDate").text if check_exists_by_xpath("//xpath/to/appealDate") else "")
        AppealedBy.append(browser.find_element("xpath", "//xpath/to/appealedBy").text if check_exists_by_xpath("//xpath/to/appealedBy") else "")
        WritofEvictionIssuedDate.append(browser.find_element("xpath", "//xpath/to/writofEvictionIssuedDate").text if check_exists_by_xpath("//xpath/to/writofEvictionIssuedDate") else "")
        WritofFieriFaciasIssuedDate.append(browser.find_element("xpath", "//xpath/to/writofFieriFaciasIssuedDate").text if check_exists_by_xpath("//xpath/to/writofFieriFaciasIssuedDate") else "")
        Plaintiff1Name.append(browser.find_element("xpath", "//xpath/to/plaintiff1Name").text if check_exists_by_xpath("//xpath/to/plaintiff1Name") else "")
        Plaintiff1DBATA.append(browser.find_element("xpath", "//xpath/to/plaintiff1DBATA").text if check_exists_by_xpath("//xpath/to/plaintiff1DBATA") else "")
        Plaintiff1Address.append(browser.find_element("xpath", "//xpath/to/plaintiff1Address").text if check_exists_by_xpath("//xpath/to/plaintiff1Address") else "")
        Plaintiff1Attorney.append(browser.find_element("xpath", "//xpath/to/plaintiff1Attorney").text if check_exists_by_xpath("//xpath/to/plaintiff1Attorney") else "")
        Plaintiff2Name.append(browser.find_element("xpath", "//xpath/to/plaintiff2Name").text if check_exists_by_xpath("//xpath/to/plaintiff2Name") else "")
        Plaintiff2DBATA.append(browser.find_element("xpath", "//xpath/to/plaintiff2DBATA").text if check_exists_by_xpath("//xpath/to/plaintiff2DBATA") else "")
        Plaintiff2Address.append(browser.find_element("xpath", "//xpath/to/plaintiff2Address").text if check_exists_by_xpath("//xpath/to/plaintiff2Address") else "")
        Plaintiff2Attorney.append(browser.find_element("xpath", "//xpath/to/plaintiff2Attorney").text if check_exists_by_xpath("//xpath/to/plaintiff2Attorney") else "")
        Plaintiff3Name.append(browser.find_element("xpath", "//xpath/to/plaintiff3Name").text if check_exists_by_xpath("//xpath/to/plaintiff3Name") else "")
        Plaintiff3DBATA.append(browser.find_element("xpath", "//xpath/to/plaintiff3DBATA").text if check_exists_by_xpath("//xpath/to/plaintiff3DBATA") else "")
        Plaintiff3Address.append(browser.find_element("xpath", "//xpath/to/plaintiff3Address").text if check_exists_by_xpath("//xpath/to/plaintiff3Address") else "")
        Plaintiff3Attorney.append(browser.find_element("xpath", "//xpath/to/plaintiff3Attorney").text if check_exists_by_xpath("//xpath/to/plaintiff3Attorney") else "")
        Defendant1Name.append(browser.find_element("xpath", "//xpath/to/defendant1Name").text if check_exists_by_xpath("//xpath/to/defendant1Name") else "")
        Defendant1DBATA.append(browser.find_element("xpath", "//xpath/to/defendant1DBATA").text if check_exists_by_xpath("//xpath/to/defendant1DBATA") else "")
        Defendant1Address.append(browser.find_element("xpath", "//xpath/to/defendant1Address").text if check_exists_by_xpath("//xpath/to/defendant1Address") else "")
        Defendant1Attorney.append(browser.find_element("xpath", "//xpath/to/defendant1Attorney").text if check_exists_by_xpath("//xpath/to/defendant1Attorney") else "")
        Defendant2Name.append(browser.find_element("xpath", "//xpath/to/defendant2Name").text if check_exists_by_xpath("//xpath/to/defendant2Name") else "")
        Defendant2DBATA.append(browser.find_element("xpath", "//xpath/to/defendant2DBATA").text if check_exists_by_xpath("//xpath/to/defendant2DBATA") else "")
        Defendant2Address.append(browser.find_element("xpath", "//xpath/to/defendant2Address").text if check_exists_by_xpath("//xpath/to/defendant2Address") else "")
        Defendant2Attorney.append(browser.find_element("xpath", "//xpath/to/defendant2Attorney").text if check_exists_by_xpath("//xpath/to/defendant2Attorney") else "")
        Defendant3Name.append(browser.find_element("xpath", "//xpath/to/defendant3Name").text if check_exists_by_xpath("//xpath/to/defendant3Name") else "")
        Defendant3DBATA.append(browser.find_element("xpath", "//xpath/to/defendant3DBATA").text if check_exists_by_xpath("//xpath/to/defendant3DBATA") else "")
        Defendant3Address.append(browser.find_element("xpath", "//xpath/to/defendant3Address").text if check_exists_by_xpath("//xpath/to/defendant3Address") else "")
        Defendant3Attorney.append(browser.find_element("xpath", "//xpath/to/defendant3Attorney").text if check_exists_by_xpath("//xpath/to/defendant3Attorney") else "")
        Defendant4Name.append(browser.find_element("xpath", "//xpath/to/defendant4Name").text if check_exists_by_xpath("//xpath/to/defendant4Name") else "")
        Defendant4DBATA.append(browser.find_element("xpath", "//xpath/to/defendant4DBATA").text if check_exists_by_xpath("//xpath/to/defendant4DBATA") else "")
        Defendant4Address.append(browser.find_element("xpath", "//xpath/to/defendant4Address").text if check_exists_by_xpath("//xpath/to/defendant4Address") else "")
        Defendant4Attorney.append(browser.find_element("xpath", "//xpath/to/defendant4Attorney").text if check_exists_by_xpath("//xpath/to/defendant4Attorney") else "")
        Defendant5Name.append(browser.find_element("xpath", "//xpath/to/defendant5Name").text if check_exists_by_xpath("//xpath/to/defendant5Name") else "")
        Defendant5DBATA.append(browser.find_element("xpath", "//xpath/to/defendant5DBATA").text if check_exists_by_xpath("//xpath/to/defendant5DBATA") else "")
        Defendant5Address.append(browser.find_element("xpath", "//xpath/to/defendant5Address").text if check_exists_by_xpath("//xpath/to/defendant5Address") else "")
        Defendant5Attorney.append(browser.find_element("xpath", "//xpath/to/defendant5Attorney").text if check_exists_by_xpath("//xpath/to/defendant5Attorney") else "")
        Defendant6Name.append(browser.find_element("xpath", "//xpath/to/defendant6Name").text if check_exists_by_xpath("//xpath/to/defendant6Name") else "")
        Defendant6DBATA.append(browser.find_element("xpath", "//xpath/to/defendant6DBATA").text if check_exists_by_xpath("//xpath/to/defendant6DBATA") else "")
        Defendant6Address.append(browser.find_element("xpath", "//xpath/to/defendant6Address").text if check_exists_by_xpath("//xpath/to/defendant6Address") else "")
        Defendant6Attorney.append(browser.find_element("xpath", "//xpath/to/defendant6Attorney").text if check_exists_by_xpath("//xpath/to/defendant6Attorney") else "")
        Next_Hearing_Date.append(browser.find_element("xpath", "//xpath/to/nextHearingDate").text if check_exists_by_xpath("//xpath/to/nextHearingDate") else "")
        Next_Hearing_Time.append(browser.find_element("xpath", "//xpath/to/nextHearingTime").text if check_exists_by_xpath("//xpath/to/nextHearingTime") else "")
        Admin_tag.append(browser.find_element("xpath", "//xpath/to/adminTag").text if check_exists_by_xpath("//xpath/to/adminTag") else "")
    except Exception as e:
        logger.error(f"Error scraping case {case_num}: {e}")
        # Append empty values for all fields if scraping fails
        for lst in [CaseNumber, FiledDate, CaseType, DebtType, Judgment, Costs, AttorneyFees, PrincipalAmount, OtherAmount, InterestAward, Possession, WritIssuedDate, HomesteadExemptionWaived, IsJudgmentSatisfied, DateSatisfactionFiled, OtherAwarded, FurtherCaseInformation, Garnishee, Address, GarnisheeAnswer, AnswerDate, NumberofChecksReceived, AppealDate, AppealedBy, WritofEvictionIssuedDate, WritofFieriFaciasIssuedDate, Plaintiff1Name, Plaintiff1DBATA, Plaintiff1Address, Plaintiff1Attorney, Plaintiff2Name, Plaintiff2DBATA, Plaintiff2Address, Plaintiff2Attorney, Plaintiff3Name, Plaintiff3DBATA, Plaintiff3Address, Plaintiff3Attorney, Defendant1Name, Defendant1DBATA, Defendant1Address, Defendant1Attorney, Defendant2Name, Defendant2DBATA, Defendant2Address, Defendant2Attorney, Defendant3Name, Defendant3DBATA, Defendant3Address, Defendant3Attorney, Defendant4Name, Defendant4DBATA, Defendant4Address, Defendant4Attorney, Defendant5Name, Defendant5DBATA, Defendant5Address, Defendant5Attorney, Defendant6Name, Defendant6DBATA, Defendant6Address, Defendant6Attorney, Next_Hearing_Date, Next_Hearing_Time, Admin_tag]:
            lst.append("")

logger.info("Finished scraping all case details.")

In [ ]:
# Create a df of the lists
d = {'casenumber':CaseNumber,
'fileddate':FiledDate,
'casetype':CaseType,
'debttype':DebtType,
'judgment':Judgment, 
'costs':Costs,
'attorneyfees':AttorneyFees, 
'principalamount':PrincipalAmount,
'otheramount':OtherAmount,
'interestaward':InterestAward,
'possession':Possession,
'writissueddate':WritIssuedDate,
'homesteadexemptionwaived':HomesteadExemptionWaived,
'isjudgmentsatisfied': IsJudgmentSatisfied,
'datesatisfactionfiled': DateSatisfactionFiled,
'otherawarded':OtherAwarded,
'furthercaseinformation':FurtherCaseInformation,
'garnishee':Garnishee,
'address': Address,
'garnishee_answer': GarnisheeAnswer,
'answerdate': AnswerDate,
'numberofchecksreceived':NumberofChecksReceived,
'appealdate': AppealDate,
'appealedby':AppealedBy,
'writofevictionissueddate': WritofEvictionIssuedDate,
'witoffierifaciasissueddate':WritofFieriFaciasIssuedDate,
'plaintiff1name':Plaintiff1Name,
'plaintiff1dbata':Plaintiff1DBATA,
'plaintiff1address': Plaintiff1Address,
'plaintiff1attorney': Plaintiff1Attorney, 
'plaintiff2name': Plaintiff2Name,
'plaintiff2dbata':Plaintiff2DBATA, 
'plaintiff2address':Plaintiff2Address,
'plaintiff2attorney':Plaintiff2Attorney,
'plaintiff3name':Plaintiff3Name,
'plaintiff3dbata':Plaintiff3DBATA,
'plaintiff3address':Plaintiff3Address,
'plaintiff3attorney':Plaintiff3Attorney,
'defendant1name':Defendant1Name,
'defendant1dbata':Defendant1DBATA,
'defendant1address':Defendant1Address,
'defendant1attorney':Defendant1Attorney,
'defendant2name':Defendant2Name,
'defendant2dbata':Defendant2DBATA,
'defendant2address':Defendant2Address,
'defendant2attorney':Defendant2Attorney,
'defendant3name':Defendant3Name,
'defendant3dbata':Defendant3DBATA,
'defendant3address':Defendant3Address,
'defendant3attorney':Defendant3Attorney,
'defendant4name':Defendant4Name,
'defendant4dbata':Defendant4DBATA,
'defendant4address':Defendant4Address,
'defendant4attorney':Defendant4Attorney,
'defendant5name':Defendant5Name,
'defendant5dbata':Defendant5DBATA,
'defendant5address':Defendant5Address,
'defendant5attorney':Defendant5Attorney,
'defendant6name':Defendant6Name,
'defendant6dbata':Defendant6DBATA,
'defendant6address':Defendant6Address,
'defendant6attorney':Defendant6Attorney, 
'next_hearing_date':Next_Hearing_Date,
'next_hearing_time':Next_Hearing_Time, 
    'administrativehearing':Admin_tag}

df_detail = pd.DataFrame(d)

df_detail = df_detail[df_detail['casenumber'].str.strip().astype(bool)] #remove empty casenumber rows

TodaysDate = pd.to_datetime('today').date()
#TodaysDate = time.strftime("%d-%m-%Y")

df_detail['date_collected'] = TodaysDate

TodaysDate = time.strftime("%d-%m-%Y")

# Format all date columns to YYYY-MM-DD format
date_columns = ['fileddate', 'writissueddate', 'datesatisfactionfiled', 'answerdate', 
                'appealdate', 'writofevictionissueddate', 'witoffierifaciasissueddate', 
                'next_hearing_date', 'date_collected']

for col in date_columns:
    if col in df_detail.columns:
        # Convert to datetime, handling empty strings and NaN values
        df_detail[col] = pd.to_datetime(df_detail[col], errors='coerce')
        # Format as YYYY-MM-DD string
        df_detail[col] = df_detail[col].dt.strftime('%Y-%m-%d')

# Export to detail file
df_detail.to_csv(os.path.join(OUTDIR, f"detail_{TodaysDate}.csv"), index=False)

# Export summary to file
#UD_days.to_csv(os.path.join(OUTDIR, f"summary_{TodaysDate}.csv"), index=False)


In [ ]:
df_detail.tail()

,CaseNumber,FiledDate,CaseType,DebtType,Judgment,Costs,AttorneyFees,PrincipalAmount,OtherAmount,InterestAward,...,Defendant5Address,Defendant5Attorney,Defendant6Name,Defendant6DBATA,Defendant6Address,Defendant6Attorney,Next_Hearing_Date,Next_Hearing_Time,AdministrativeHearing,Date_Collected
1303,GV25006832-00,09/12/2025,Unlawful Detainer,,,,,,,,...,,,,,,,09/30/2025,01:30 PM,False,2025-09-13
1304,GV25006833-00,09/12/2025,Unlawful Detainer,,,,,,,,...,,,,,,,09/30/2025,01:30 PM,False,2025-09-13
1305,GV25006835-00,09/12/2025,Unlawful Detainer,,,,,,,,...,,,,,,,09/30/2025,01:30 PM,False,2025-09-13
1306,GV25006836-00,09/12/2025,Unlawful Detainer,,,,,,,,...,,,,,,,09/30/2025,01:30 PM,False,2025-09-13
1307,GV25006838-00,09/12/2025,Unlawful Detainer,,,,,,,,...,,,,,,,09/30/2025,01:30 PM,False,2025-09-13
